In [ ]:
import os
import yaml
import pandas as pd

# === CONFIG ===
PROJECTS_DIR = r"C:\Users\Admin\OneDrive\Education\Master of Info - Thesis\Mobile App Data\Config Files"  # desktop
OUTPUT_DIR = r"C:\GitHub\Android-Mobile-Apps"
os.makedirs(OUTPUT_DIR, exist_ok=True)
OUTPUT_CSV = os.path.join(OUTPUT_DIR, "instrumentation_test_summary.csv")

# === CLASSIFICATION KEYWORDS ===
TEST_TYPES = {
    #'firebase_test_lab': ['firebase test', 'gcloud firebase test android run'],
    'firebase_Full':['gcloud firebase test android run'],
    'firebase_Compact':['Firebase-Test-Lab-Action'],
    'appcenter_test': ['appcenter test run', 'microsoft/appcenter-test-cli-action'],
    'browserstack_test': ['browserstack', 'browserstack/github-actions'],
    'GitHub_emulator_full': ['android-emulator-runner'],
    'GitHub_emulator_compact':['malinskiy/action-android/emulator-run-cmd'],
    'GitHub_emulator_manual':['create avd'],
    'GitHub_GMD':['cleanManagedDevices'],
    #'GitHub_gradle':['connectedReleaseAndroidTest','connectedcheck', 'connectedDebugAndroidTest','connectedAndroidTest'],
    'Unit_Test': ['gradlew test', './gradlew test', 'testDebugUnitTest', 'testReleaseUnitTest',
                    'test', 'run unit tests', 'run: test', 'npm test', 'yarn test'],
    'Other':['instrumentation','instrument']
    #'GitHub_avd':['adb', 'avdmanager']
    
}

# === RESULTS STRUCTURE ===
project_results = {}

# === DETECTION LOGIC ===
def detect_testing_types(yaml_text):
    # Remove commented lines
    uncommented_text = '\n'.join(
        line for line in yaml_text.splitlines()
        if not line.strip().startswith('#')
    ).lower()

    found = set()
    for label, keywords in TEST_TYPES.items():
        for kw in keywords:
            if kw.lower() in uncommented_text:
                found.add(label)
    return found


# === MAIN PARSER ===
def parse_yaml_file(file_path):
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            raw = f.read().replace('\t', ' ')
            detected = detect_testing_types(raw)  # <== Move this BEFORE parsing
            content = yaml.safe_load(raw)  # Parse just to check for validity
            if not content:
                return {'types': detected, 'error': True}
            return {'types': detected, 'error': False}
    except Exception as e:
        return {'types': set(), 'error': True}


# === PROJECT SCANNER ===
for root, _, files in os.walk(PROJECTS_DIR):
    for file in files:
        if file.endswith(('.yml', '.yaml')):
            file_path = os.path.join(root, file)
#            print(f"📄 Scanning: {file_path}")

            filename = os.path.basename(file_path)
            parts = filename.split(".")
            project_name = (parts[1] if len(parts) > 2 else parts[0]).lower() # Between first and second dot

            result = parse_yaml_file(file_path)
#            print(f"→ Project: {project_name}, Test Types: {result['types'] or 'none'}, YAML Error: {result['error']}")

            if project_name not in project_results:
                project_results[project_name] = {'types': set(), 'errors': 0}

            project_results[project_name]['types'].update(result['types'])
            if result['error']:
                project_results[project_name]['errors'] += 1

# === EXPORT CSV ===
rows = []
for project, result in project_results.items():
    # Remove 'Other' if there's at least one other test type
    cleaned_types = result['types']
    if 'Other' in cleaned_types and len(cleaned_types) > 1:
        cleaned_types = cleaned_types - {'Other'}

    rows.append({
        'project': project,
        'test_types': ', '.join(sorted(cleaned_types)) if cleaned_types else 'none',
        'yaml_errors': result['errors']
    })

df = pd.DataFrame(rows)
df.to_csv(OUTPUT_CSV, index=False)

# === SAVE TO DATAFRAME ===
Total_Projects = df.copy()

print(f"\n✅ Summary written to: {OUTPUT_CSV}")






# Part two - Make the list for all projects

import pandas as pd

PROJECTS_DIR = r"C:\Users\Admin\OneDrive\Education\Master of Info - Thesis\Config Files"  # desktop
OUTPUT_DIR = r"C:\GitHub\Android-Mobile-Apps"
os.makedirs(OUTPUT_DIR, exist_ok=True)
OUTPUT_CSV = os.path.join(OUTPUT_DIR, "Total_Project_v1.0.csv")

#read from table above
df = Total_Projects.copy()

# 0 - Unit Test: True if any test_type include "Unit_Test"
df['Unit Test'] = df['test_types'].apply(
    lambda x: any(item.strip().startswith('Unit_Test') for item in x.split(','))
)


# 1 - Instrumentation Testing: True if test_type is not "none" nor only unit_test is detected
df['Instrumentation Testing'] = df['test_types'].apply(
    lambda x: x.strip().lower() != 'none' and not (
        len([t for t in x.split(',') if t.strip()]) == 1 and x.strip() == 'Unit_Test'
    )
)

# 2 - GitHub Action: True if any test_type starts with "GitHub"
df['GitHub Action'] = df['test_types'].apply(
    lambda x: any(item.strip().startswith('GitHub') for item in x.split(','))
)

# 3 - GitHub Action Type: include all GitHub-related test types
df['GitHub Action Type'] = df['test_types'].apply(
    lambda x: ', '.join([item.strip() for item in x.split(',') if item.strip().startswith('GitHub')])
)

# 4 - Third_Party: True if any test_type does not start with "GitHub" and is not "none" or "other" or "unit_test"
df['Third_Party'] = df['test_types'].apply(
    lambda x: any(
        not item.strip().startswith('GitHub') and item.strip().lower() not in ['none', 'other','unit_test']
        for item in x.split(',')
    )
)

# 5 - Third_Party_Name: list the test types that do not start with "GitHub" and are not "none" or "other" or "unit_test"
df['Third_Party_Name'] = df['test_types'].apply(
    lambda x: ', '.join([
        item.strip() for item in x.split(',')
        if not item.strip().startswith('GitHub') and item.strip().lower() not in ['none', 'other','unit_test']
    ])
)


# Save it back to CSV if needed

df.to_csv(OUTPUT_CSV, index=False)

print(f"\n✅ Summary written to: {OUTPUT_CSV}")


# Optional: assign to Total_Projects for in-memory use
Total_Projects = df





✅ Summary written to: C:\GitHub\Android-Mobile-Apps\instrumentation_test_summary.csv


In [1]:
import os
import yaml
import pandas as pd

# === CONFIG ===
PROJECTS_DIR = r"C:\Users\Admin\OneDrive\Education\Master of Info - Thesis\Config Files"
OUTPUT_DIR = r"C:\GitHub\Android-Mobile-Apps"
os.makedirs(OUTPUT_DIR, exist_ok=True)

SUMMARY_CSV = os.path.join(OUTPUT_DIR, "project_test_types.csv")
DETAILED_CSV = os.path.join(OUTPUT_DIR, "project_api_levels_detailed.csv")

# === TEST CLASSIFICATION KEYWORDS ===
TEST_TYPES = {
    #'firebase_test_lab': ['firebase test', 'gcloud firebase test android run'],
    'firebase_Full':['gcloud firebase test android run'],
    'firebase_Compact':['Firebase-Test-Lab-Action'],
    'appcenter_test': ['appcenter test run', 'microsoft/appcenter-test-cli-action'],
    'browserstack_test': ['browserstack', 'browserstack/github-actions'],
    'GitHub_emulator_full': ['android-emulator-runner'],
    'GitHub_emulator_compact':['malinskiy/action-android/emulator-run-cmd'],
    'GitHub_emulator_manual':['create avd'],
    'GitHub_GMD':['cleanManagedDevices'],
    #'GitHub_gradle':['connectedReleaseAndroidTest','connectedcheck', 'connectedDebugAndroidTest','connectedAndroidTest'],
    'Unit_Test': ['gradlew test', './gradlew test', 'testDebugUnitTest', 'testReleaseUnitTest',
                    'test', 'run unit tests', 'run: test', 'npm test', 'yarn test'],
    'Other':['instrumentation','instrument']
    #'GitHub_avd':['adb', 'avdmanager']
}

# === DETECT TEST TYPES (ignoring comments) ===
def detect_testing_types(yaml_text):
    uncommented_text = '\n'.join(
        line for line in yaml_text.splitlines()
        if not line.strip().startswith('#')
    ).lower()
    found = set()
    for label, keywords in TEST_TYPES.items():
        for kw in keywords:
            if kw in uncommented_text:
                found.add(label)
    return found

# === EXTRACT MATRIX + HARDCODED API LEVELS ===
def extract_api_levels_precise(obj):
    matrix_api_levels = set()
    hardcoded_api_levels = set()

    def recurse(o, parent_key=None):
        if isinstance(o, dict):
            for k, v in o.items():
                key_lower = k.lower() if isinstance(k, str) else ""
                if key_lower == 'api-level':
                    if isinstance(v, list):
                        matrix_api_levels.update(str(val) for val in v if str(val).isdigit())
                    elif isinstance(v, (int, str)) and str(v).isdigit():
                        hardcoded_api_levels.add(str(v))
                else:
                    recurse(v, k)
        elif isinstance(o, list):
            for item in o:
                recurse(item, parent_key)

    recurse(obj)
    # Remove matrix values from hardcoded list
    return matrix_api_levels, hardcoded_api_levels - matrix_api_levels

# === PARSE YAML FILE ===
def parse_yaml_file(file_path):
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            raw = f.read().replace('\t', ' ')
            test_types = detect_testing_types(raw)
            content = yaml.safe_load(raw)
            if not content:
                return {'types': test_types, 'matrix_apis': set(), 'hardcoded_apis': set(), 'error': True}
            matrix_apis, hardcoded_apis = extract_api_levels_precise(content.get('jobs', {}))
            return {'types': test_types, 'matrix_apis': matrix_apis, 'hardcoded_apis': hardcoded_apis, 'error': False}
    except Exception:
        return {'types': set(), 'matrix_apis': set(), 'hardcoded_apis': set(), 'error': True}

# === SCAN PROJECTS ===
project_results = {}
detailed_rows = {}

for root, _, files in os.walk(PROJECTS_DIR):
    for file in files:
        if file.endswith(('.yml', '.yaml')):
            file_path = os.path.join(root, file)
            filename = os.path.basename(file_path)
            parts = filename.split(".")
            project_name = parts[1] if len(parts) > 2 else parts[0]

            result = parse_yaml_file(file_path)

            if project_name not in project_results:
                project_results[project_name] = {
                    'types': set(),
                    'matrix_api_levels': set(),
                    'hardcoded_api_levels': set(),
                    'errors': 0,
                    'yml_count': 0
                }
                detailed_rows[project_name] = []

            project_results[project_name]['types'].update(result['types'])
            project_results[project_name]['matrix_api_levels'].update(result['matrix_apis'])
            project_results[project_name]['hardcoded_api_levels'].update(result['hardcoded_apis'])
            project_results[project_name]['yml_count'] += 1
            if result['error']:
                project_results[project_name]['errors'] += 1

# === BUILD DETAILED ROWS WITH YAML COUNT ===
final_detailed_rows = []
for project, data in project_results.items(): 
    for api in data['matrix_api_levels']:
        final_detailed_rows.append({
            'project': project,
            'api_level': api,
            'source': 'matrix',
            'yml_count': data['yml_count']
        })
    for api in data['hardcoded_api_levels']:
        final_detailed_rows.append({
            'project': project,
            'api_level': api,
            'source': 'hardcoded',
            'yml_count': data['yml_count']
        })

# === EXPORT SUMMARY CSV ===
summary_rows = []
for project, result in project_results.items():
    summary_rows.append({
        'project': project,
        'test_types': ', '.join(sorted(result['types'])) if result['types'] else 'none',
        'distinct_matrix_api_levels': len(result['matrix_api_levels']),
        'distinct_hardcoded_api_levels': len(result['hardcoded_api_levels']),
        'yml_count': result['yml_count'],
        'yaml_errors': result['errors']
    })

pd.DataFrame(summary_rows).to_csv(SUMMARY_CSV, index=False)
pd.DataFrame(final_detailed_rows).to_csv(DETAILED_CSV, index=False)

print(f"\n✅ Summary CSV saved to: {SUMMARY_CSV}")
print(f"✅ Detailed CSV saved to: {DETAILED_CSV}")



✅ Summary CSV saved to: C:\GitHub\Android-Mobile-Apps\project_test_types.csv
✅ Detailed CSV saved to: C:\GitHub\Android-Mobile-Apps\project_api_levels_detailed.csv


In [7]:
import pandas as pd
# Opt-in to the future behavior to silence warning
pd.set_option('future.no_silent_downcasting', True)
# === Input Paths ===
main_path = r"C:\GitHub\Android-Mobile-Apps\Total_Project_v1.0.csv"
api_detail_path = r"C:\GitHub\Android-Mobile-Apps\project_api_levels_detailed.csv"
output_path = r"C:\GitHub\Android-Mobile-Apps\Total_Project_v1.2_with_api_flags.csv"

# === Load Data ===
df_main = pd.read_csv(main_path)
df_api = pd.read_csv(api_detail_path)

# === Clean project names for matching ===
df_main['project'] = df_main['project'].str.strip().str.lower()
df_api['project'] = df_api['project'].str.strip().str.lower()

# === Define Target APIs ===
target_apis = [str(api) for api in [16, 19, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33]]

# === Create a pivot table to flag API usage ===
df_api['api_level'] = df_api['api_level'].astype(str)
api_flags = df_api[df_api['api_level'].isin(target_apis)].pivot_table(
    index='project',
    columns='api_level',
    aggfunc='size',
    fill_value=0
).astype(bool).reset_index()

# === Rename API columns to API_XX format
api_flags.columns = ['project'] + [f'API_{col}' for col in api_flags.columns if col != 'project']

# === Merge with Main Data ===
df_merged = df_main.merge(api_flags, on='project', how='left')

# === Fill missing API flags with False
for api in target_apis:
    col = f'API_{api}'
    if col not in df_merged.columns:
        df_merged[col] = False
api_cols = [f'API_{api}' for api in target_apis]
df_merged[api_cols] = df_merged[api_cols].fillna(False).infer_objects(copy=False)

df_merged[api_cols] = df_merged[api_cols].infer_objects(copy=False).astype(bool)


# === Add Total API count
df_merged['Total API'] = df_merged[[f'API_{api}' for api in target_apis]].sum(axis=1)

# === Export to CSV
df_merged.to_csv(output_path, index=False)
print(f"✅ Updated CSV saved to: {output_path}")


✅ Updated CSV saved to: C:\GitHub\Android-Mobile-Apps\Total_Project_v1.2_with_api_flags.csv
